# Chain-of-Thought Prompting Elicits Reasoning in Language Models

## Learning Objectives
1. Understand why step-by-step reasoning improves LLM accuracy
2. Implement zero-shot and few-shot chain-of-thought prompting
3. Analyze reasoning chains to identify correct vs incorrect paths
4. Compare direct answering vs CoT on reasoning-heavy tasks
5. Optimize CoT for production use (latency vs accuracy trade-offs)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import re
import time

# Device setup
device = "cpu"  # No GPU needed for this notebook
np.random.seed(42)

print("Chain-of-Thought Prompting Notebook")
print("=" * 50)

## Level 1: Basic Chain-of-Thought on Simple Math

Direct problem-solving vs. reasoning-chain approach.

In [ ]:
def simulate_direct_answer(question: str) -> str:
    """Model attempts to answer directly without reasoning."""
    if "how many" in question.lower():
        if np.random.random() > 0.3:
            return "6"  # Correct for simple example
        else:
            return "8"  # Wrong answer
    return "Unknown"

def simulate_cot_answer(question: str) -> str:
    """Model generates step-by-step reasoning before answering."""
    reasoning = "Let's think step by step.\\n"
    
    if "apples" in question.lower():
        reasoning += "Step 1: Sarah starts with 5 apples.\\n"
        reasoning += "Step 2: She buys 3 more, so 5 + 3 = 8 apples.\\n"
        reasoning += "Step 3: She gives 2 to her friend, so 8 - 2 = 6.\\n"
        reasoning += "Answer: Sarah has 6 apples."
        return reasoning
    
    return reasoning + "Unable to reason about this problem."

# Test basic CoT
question = "Sarah has 5 apples. She buys 3 more and gives 2 to her friend. How many apples does she have?"

print("Question:", question)
print()
print("Direct Answer (without reasoning):")
print(simulate_direct_answer(question))
print()
print("With Chain-of-Thought:")
print(simulate_cot_answer(question))

## Level 2: Advanced - Multiple Problems, Accuracy Comparison

Measuring accuracy with and without chain-of-thought on diverse problems.

In [ ]:
# Test suite of reasoning problems
test_problems = [
    {"question": "If a store sells 10 widgets on day 1 and 15 on day 2, how many total?", "answer": "25", "category": "arithmetic"},
    {"question": "Alice is older than Bob, Bob is older than Charlie. Who is oldest?", "answer": "Alice", "category": "logic"},
    {"question": "A box contains 3 red, 2 blue, 1 green ball. What colors are possible?", "answer": "red, blue, or green", "category": "logic"},
    {"question": "John bought 5 books at $10 each and 3 notebooks at $2 each. How much total?", "answer": "56", "category": "multi_step"},
    {"question": "If 5 workers complete task in 10 days, how many days for 10 workers?", "answer": "5", "category": "reasoning"}
]

def simulate_accuracy(problem_type, use_cot=False):
    """Simulate model accuracy based on problem type and CoT usage."""
    base_accuracy = {
        "arithmetic": 0.85 if use_cot else 0.60,
        "logic": 0.75 if use_cot else 0.40,
        "multi_step": 0.80 if use_cot else 0.35,
        "reasoning": 0.85 if use_cot else 0.45
    }
    return base_accuracy.get(problem_type, 0.5)

# Compare accuracy across problem types
categories = set(p["category"] for p in test_problems)
direct_accuracy = {}
cot_accuracy = {}

for category in sorted(categories):
    direct = simulate_accuracy(category, use_cot=False)
    cot = simulate_accuracy(category, use_cot=True)
    direct_accuracy[category] = direct
    cot_accuracy[category] = cot
    improvement = ((cot - direct) / direct) * 100
    print(f"{category:15} | Direct: {direct:.1%} | CoT: {cot:.1%} | Improvement: +{improvement:.0f}%")

print()
print("Key observation: CoT provides 20-50% improvement across all problem types!")

## Real-World Example 1: Few-Shot Chain-of-Thought

Training the model with examples before asking a new question.

In [ ]:
def few_shot_cot_prompt(examples: list, question: str) -> str:
    """Construct a few-shot CoT prompt with examples."""
    prompt = ""
    
    for i, example in enumerate(examples, 1):
        prompt += f"Example {i}:\\n"
        prompt += f"Question: {example['question']}\\n"
        prompt += "Let's think step by step.\\n"
        prompt += example['reasoning'] + "\\n"
        prompt += f"Answer: {example['answer']}\\n\\n"
    
    prompt += f"Question: {question}\\n"
    prompt += "Let's think step by step.\\n"
    
    return prompt

# Define examples with reasoning
examples = [
    {
        "question": "If a recipe calls for 2 cups flour and serves 4 people, how much flour for 8 people?",
        "reasoning": "Step 1: The recipe serves 4 people.\\nStep 2: We want to serve 8 people.\\nStep 3: 8 / 4 = 2, so we need 2x the recipe.\\nStep 4: 2 cups * 2 = 4 cups flour.",
        "answer": "4 cups"
    },
    {
        "question": "Three friends split a $30 dinner bill equally. One adds a $6 tip. What does each pay?",
        "reasoning": "Step 1: Total bill is $30.\\nStep 2: Tip is $6, total with tip = $36.\\nStep 3: Three friends split $36.\\nStep 4: $36 / 3 = $12 per person.",
        "answer": "$12"
    }
]

# New question
new_question = "A store has 100 items. It receives a shipment of 40 items and sells 25. How many items does it have?"

prompt = few_shot_cot_prompt(examples, new_question)
print("Few-Shot CoT Prompt:")
print("=" * 60)
print(prompt)
print()
print("Expected reasoning and answer:")
print("Step 1: Store starts with 100 items.")
print("Step 2: Receives 40, so 100 + 40 = 140 items.")
print("Step 3: Sells 25, so 140 - 25 = 115 items.")
print("Answer: 115 items")

## Real-World Example 2: Self-Consistency with Multiple Chains

Generate multiple reasoning chains and vote on the answer.

In [ ]:
def generate_reasoning_chains(question: str, num_chains: int = 5) -> list:
    """Generate multiple reasoning chains (simulated)."""
    chains = []
    
    base_reasoning = [
        "Step 1: Identify the problem type.\\nStep 2: Break into smaller parts.\\nStep 3: Solve each part.\\nStep 4: Combine results.",
        "First, let's understand what we know.\\nThen determine what we need to find.\\nApply relevant formulas or logic.\\nVerify the answer makes sense.",
        "Start with initial conditions.\\nApply operations in order given.\\nTrack intermediate values.\\nArrive at final answer.",
        "Think about the relationships between quantities.\\nApply mathematical operations correctly.\\nCheck units and magnitude.\\nState the answer clearly.",
        "Decompose complex problem into steps.\\nEvaluate each step independently.\\nCombine to get final result.\\nValidate reasonableness."
    ]
    
    for i in range(num_chains):
        reasoning = base_reasoning[i % len(base_reasoning)]
        if i < 3:
            answer = "Answer: 115"  # Correct
        else:
            answer = "Answer: 115" if np.random.random() > 0.3 else "Answer: 114"
        
        chains.append({
            "chain": reasoning,
            "answer": answer
        })
    
    return chains

def vote_on_answer(chains: list) -> dict:
    """Extract and vote on answers from reasoning chains."""
    answers = [c["answer"].split(":")[-1].strip() for c in chains]
    answer_counts = Counter(answers)
    most_common = answer_counts.most_common(1)[0]
    
    return {
        "best_answer": most_common[0],
        "confidence": most_common[1] / len(chains),
        "vote_counts": dict(answer_counts),
        "chains": chains
    }

question = "A store starts with 100 items, receives 40, and sells 25. How many items remain?"
chains = generate_reasoning_chains(question, num_chains=5)
result = vote_on_answer(chains)

print("Self-Consistency Results:")
print("=" * 60)
print(f"Question: {question}")
print()
print(f"Generated {len(chains)} reasoning chains:")
for i, chain in enumerate(chains, 1):
    print(f"  Chain {i}: {chain['answer']}")
print()
print(f"Vote Results: {result['vote_counts']}")
print(f"Best Answer: {result['best_answer']}")
print(f"Confidence: {result['confidence']:.1%}")
print()
print("Observation: Multiple chains vote on answer 115, high confidence!")

## Key Takeaways

**Core Insight:** Step-by-step reasoning dramatically improves language model performance on complex tasks.

### Why Chain-of-Thought Works
| Reason | Explanation |
|--------|-------------|
| **Extended Compute** | More tokens = more chances to find correct answer |
| **Transparency** | Models can self-correct during reasoning |
| **Activation** | Intermediate steps activate correct reasoning patterns |
| **Decomposition** | Complex problems become manageable subproblems |

### CoT Approaches
| Method | Accuracy | Latency | Setup |
|--------|----------|---------|-------|
| Direct Answer | 40-60% | Very fast | None |
| Zero-shot CoT | 60-80% | Moderate | Just add instruction |
| Few-shot CoT | 70-85% | Moderate | Need examples |
| Self-Consistency | 75-90% | High | Generate multiple chains |

### When to Use What
- **Zero-shot:** Exploring new domains, quick iteration
- **Few-shot:** Production tasks with good examples available
- **Self-consistency:** High-stakes decisions, accuracy > latency
- **Direct:** Simple lookup questions, latency critical

### Production Considerations
- CoT adds 2-10x to inference latency (longer generation)
- Use shorter reasoning formats to reduce token cost
- Combine with caching for frequently asked questions
- Monitor accuracy improvement vs latency trade-off
- For simple tasks (knowledge lookup), skip CoT entirely

### Related Concepts
- [ReAct](./02-react.md) — Add tool use to reasoning
- [Tree of Thoughts](./03-tree-of-thoughts.md) — Explore multiple branches

In [ ]:
# Visualization: CoT impact on accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Accuracy improvement by category
categories = list(sorted(direct_accuracy.keys()))
x_pos = np.arange(len(categories))
width = 0.35

direct_scores = [direct_accuracy[c] * 100 for c in categories]
cot_scores = [cot_accuracy[c] * 100 for c in categories]

axes[0].bar(x_pos - width/2, direct_scores, width, label='Direct Answer', color='steelblue', alpha=0.8)
axes[0].bar(x_pos + width/2, cot_scores, width, label='Chain-of-Thought', color='darkorange', alpha=0.8)
axes[0].set_xlabel('Problem Category', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
axes[0].set_title('CoT Improves Accuracy Across Problem Types', fontsize=13, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(categories, rotation=45, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim([0, 100])

# Plot 2: Latency vs accuracy trade-off
methods = ['Direct', 'Zero-shot CoT', 'Few-shot CoT', 'Self-Consistency']
accuracies = [50, 70, 78, 82]
latencies = [0.2, 2.0, 2.5, 10.0]
colors = ['steelblue', 'orange', 'darkorange', 'red']

for method, acc, lat, color in zip(methods, accuracies, latencies, colors):
    axes[1].scatter(lat, acc, s=300, alpha=0.7, color=color, edgecolors='black', linewidth=2)
    axes[1].annotate(method, (lat, acc), xytext=(5, 5), textcoords='offset points', fontsize=10, fontweight='bold')

axes[1].set_xlabel('Relative Latency', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
axes[1].set_title('Accuracy vs Latency Trade-off', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([-1, 12])
axes[1].set_ylim([40, 90])

plt.tight_layout()
plt.show()

print("Visualization complete.")
print()
print("Summary Statistics:")
avg_improvement = sum((cot_accuracy[c] - direct_accuracy[c]) for c in categories) / len(categories) * 100
max_improvement = max((cot_accuracy[c] - direct_accuracy[c]) / direct_accuracy[c] * 100 for c in categories)
print(f"Average accuracy improvement with CoT: {avg_improvement:.1f}%")
print(f"Maximum improvement: {max_improvement:.0f}%")